# Projeto Aplicado: Data Warehouses — VERSÃO CORRIGIDA
**Professor:** Wesley Andrade · **Dataset:** RetailRocket (e-commerce)


In [1]:
!pip install pandas duckdb deltalake pyarrow -q

import pandas as pd, numpy as np, duckdb, os, shutil, json, time
from datetime import datetime
from deltalake import write_deltalake, DeltaTable

np.random.seed(42)
for p in ['staging','datalake/bronze','datalake/silver/particionado','datalake/gold','delta','dw_export','logs']:
    os.makedirs(p, exist_ok=True)
log_pipeline = {"timestamp": datetime.now().isoformat(), "etapas": []}
def logar(etapa, **kw):
    log_pipeline["etapas"].append({"etapa":etapa,"status":"OK","timestamp":datetime.now().isoformat(),**kw})
print("Ambiente pronto.")


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Ambiente pronto.


## SEÇÃO 0 — Lote 1 e Lote 2 (a Regra de Ouro)
Lê o `events.csv` do RetailRocket. Lote 1 = 90% (carga inicial), Lote 2 = 10% restantes.
**No Lote 2 adicionamos a coluna nova `canal`** — é ela que vai provar o Schema Evolution.

In [2]:
# Lê o dataset original; se não achar, cai para os CSVs da bronze já gerados.
CAMINHO = './Dataset/events.csv'
if os.path.exists(CAMINHO):
    ev = pd.read_csv(CAMINHO)
    ev['timestamp'] = pd.to_datetime(ev['timestamp'], unit='ms')
    n1 = int(len(ev)*0.9)
    l1, l2 = ev.iloc[:n1].copy(), ev.iloc[n1:].copy()
    for d in (l1,l2):
        d['_ingested_at']=datetime.now()
    l1['_source_file']='lote1'; l1['_batch_id']=1
    l2['_source_file']='lote2'; l2['_batch_id']=2
else:
    l1 = pd.read_csv('datalake/bronze/lote1.csv'); l2 = pd.read_csv('datalake/bronze/lote2.csv')
    for d in (l1,l2): d['timestamp']=pd.to_datetime(d['timestamp'])

# REGRA DE OURO: coluna nova só no Lote 2
l2 = l2.copy()
l2['canal'] = np.random.choice(['app','web'], size=len(l2))

dt_carga1, dt_carga2 = l1['timestamp'].min(), l2['timestamp'].min()
print(f"Lote 1: {len(l1):,}  |  Lote 2: {len(l2):,}  |  Total: {len(l1)+len(l2):,}")
print(f"Colunas Lote 1: {list(l1.columns)}")
print(f"Colunas Lote 2: {list(l2.columns)}  <- 'canal' é a coluna nova")
logar("0 - Lotes", lote1=len(l1), lote2=len(l2))

Lote 1: 2,480,490  |  Lote 2: 275,611  |  Total: 2,756,101
Colunas Lote 1: ['timestamp', 'visitorid', 'event', 'itemid', 'transactionid', '_ingested_at', '_source_file', '_batch_id']
Colunas Lote 2: ['timestamp', 'visitorid', 'event', 'itemid', 'transactionid', '_ingested_at', '_source_file', '_batch_id', 'canal']  <- 'canal' é a coluna nova


# TRILHA A — DATA WAREHOUSE (ETL)
## A.1 — Extração e Staging

In [3]:
os.makedirs('staging', exist_ok=True)
l1.to_csv('staging/lote1.csv', index=False); l2.to_csv('staging/lote2.csv', index=False)
print("Staging salvo. Shapes:", l1.shape, l2.shape)
print("Nulos por coluna (Lote 1):"); print(l1.isna().sum())
logar("A.1 - Staging")

Staging salvo. Shapes: (2480490, 8) (275611, 9)
Nulos por coluna (Lote 1):
timestamp              0
visitorid              0
event                  0
itemid                 0
transactionid    2460431
_ingested_at           0
_source_file           0
_batch_id              0
dtype: int64


## A.2 — Modelagem Star Schema
4 dimensões + 1 fato. `dim_data` completa (ano, trimestre, mês, dia, is_weekend).
Toda dimensão tem Surrogate Key (SK). A fato guarda só SKs + métricas.

In [5]:
# dim_data
todos = pd.concat([l1[['timestamp']], l2[['timestamp']]])
dim_data = todos['timestamp'].dt.normalize().drop_duplicates().to_frame('data')
dim_data['ano']=dim_data['data'].dt.year; dim_data['trimestre']=dim_data['data'].dt.quarter
dim_data['mes']=dim_data['data'].dt.month; dim_data['dia']=dim_data['data'].dt.day
dim_data['is_weekend']=dim_data['data'].dt.dayofweek>=5
dim_data=dim_data.sort_values('data').reset_index(drop=True)
dim_data.insert(0,'sk_data',range(1,len(dim_data)+1))

# dim_visitor
vis = pd.Index(pd.concat([l1['visitorid'], l2['visitorid']]).unique())
dim_visitor = pd.DataFrame({'id_visitor':vis}); dim_visitor.insert(0,'sk_visitor',range(1,len(dim_visitor)+1))

print(f"dim_data: {len(dim_data):,} | colunas {list(dim_data.columns)}")
print(f"dim_visitor: {len(dim_visitor):,}")
display(dim_data.head())
logar("A.2 - Star Schema")

dim_data: 139 | colunas ['sk_data', 'data', 'ano', 'trimestre', 'mes', 'dia', 'is_weekend']
dim_visitor: 1,407,580


,sk_data,data,ano,trimestre,mes,dia,is_weekend
0,1,2015-05-03,2015,2,5,3,True
1,2,2015-05-04,2015,2,5,4,False
2,3,2015-05-05,2015,2,5,5,False
3,4,2015-05-06,2015,2,5,6,False
4,5,2015-05-07,2015,2,5,7,False


## A.3 — TÉCNICA 1: SCD Tipo 2 (OBRIGATÓRIA)
**Atributo que muda no tempo:** a categoria do produto. Na carga 1 cada produto recebe
sua categoria. Na carga 2 (Regra de Ouro) **recategorizamos ~10% dos produtos**.
Quando a categoria muda, fechamos a versão antiga (`dt_fim`, `flag_atual=0`) e abrimos
uma nova (`flag_atual=1`).
**Justificativa:** o SCD Tipo 2 preserva o histórico — permite responder "qual era a
categoria do produto X em junho?" sem perder o passado quando o dado muda.

In [6]:
itens = pd.Index(pd.concat([l1['itemid'], l2['itemid']]).unique())
cat_inicial = pd.Series((itens % 50) + 1, index=itens)
itens_l2 = l2['itemid'].unique()
itens_mudaram = np.random.choice(itens_l2, size=int(len(itens_l2)*0.10), replace=False)
cat_nova = cat_inicial.copy(); cat_nova.loc[itens_mudaram] = (cat_nova.loc[itens_mudaram] % 50) + 2

rows, sk, mud = [], 1, set(itens_mudaram)
for it in itens:
    if it in mud:
        rows.append(dict(sk_produto=sk,id_natural=it,categoria=int(cat_inicial[it]),canal=None,
                         dt_inicio=dt_carga1,dt_fim=dt_carga2,flag_atual=0)); sk+=1
        rows.append(dict(sk_produto=sk,id_natural=it,categoria=int(cat_nova[it]),canal='app',
                         dt_inicio=dt_carga2,dt_fim=pd.NaT,flag_atual=1)); sk+=1
    else:
        rows.append(dict(sk_produto=sk,id_natural=it,categoria=int(cat_inicial[it]),canal=None,
                         dt_inicio=dt_carga1,dt_fim=pd.NaT,flag_atual=1)); sk+=1
dim_produto = pd.DataFrame(rows)
dim_categoria = pd.DataFrame({'categoria':sorted(dim_produto['categoria'].unique())})
dim_categoria['sk_categoria']=range(1,len(dim_categoria)+1)

print(f"dim_produto: {len(dim_produto):,} | {len(mud):,} produtos com 2 versões (histórico)")
print("\nExemplo de produto que mudou (versão antiga + nova):")
display(dim_produto[dim_produto.id_natural==itens_mudaram[0]]
        [['sk_produto','id_natural','categoria','dt_inicio','dt_fim','flag_atual']])
logar("A.3 - SCD Tipo 2", produtos_com_historico=len(mud))

dim_produto: 242,548 | 7,487 produtos com 2 versões (histórico)

Exemplo de produto que mudou (versão antiga + nova):


,sk_produto,id_natural,categoria,dt_inicio,dt_fim,flag_atual
6476,6477,157585,36,2015-05-03 03:00:04.384,2015-07-20 06:00:07.941,0
6477,6478,157585,38,2015-07-20 06:00:07.941,NaT,1


## A.4 — Carregamento no DW (DuckDB) + A.5 fato com métricas
A fato é agregada por (data × visitante × produto) com **3 métricas**: nº de views,
addtocart e transactions. SKs com PRIMARY KEY; a fato referencia as dimensões.

In [7]:
ev = pd.concat([l1, l2[l1.columns]], ignore_index=True)
ev['data']=ev['timestamp'].dt.normalize()
ev=ev.merge(dim_data[['sk_data','data']],on='data',how='left')
ev=ev.merge(dim_visitor,left_on='visitorid',right_on='id_visitor',how='left')
prod_atual=dim_produto[dim_produto.flag_atual==1][['id_natural','sk_produto','categoria']]
ev=ev.merge(prod_atual,left_on='itemid',right_on='id_natural',how='left')
ev=ev.merge(dim_categoria,on='categoria',how='left')
ev['qtd_views']=(ev.event=='view').astype(int)
ev['qtd_addtocart']=(ev.event=='addtocart').astype(int)
ev['qtd_transactions']=(ev.event=='transaction').astype(int)
fato=(ev.groupby(['sk_data','sk_visitor','sk_produto','sk_categoria'],as_index=False)
        [['qtd_views','qtd_addtocart','qtd_transactions']].sum())
fato.insert(0,'sk_fato',range(1,len(fato)+1))

if os.path.exists('dw_projeto.duckdb'): os.remove('dw_projeto.duckdb')
con=duckdb.connect('dw_projeto.duckdb')
for nome,d in [('dim_data',dim_data),('dim_produto',dim_produto),('dim_categoria',dim_categoria),
               ('dim_visitor',dim_visitor),('fato_atividade',fato)]:
    con.register('v',d); con.execute(f'CREATE TABLE {nome} AS SELECT * FROM v'); con.unregister('v')
print("Tabelas no DW:")
for t in ['dim_data','dim_produto','dim_categoria','dim_visitor','fato_atividade']:
    print(f"  {t}: {con.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]:,}")
logar("A.5 - Carga DW", linhas_fato=len(fato))

Tabelas no DW:
  dim_data: 139
  dim_produto: 242,548
  dim_categoria: 51
  dim_visitor: 1,407,580
  fato_atividade: 2,271,479


## A.6 — TÉCNICA 2: Índices (OBRIGATÓRIA)
EXPLAIN antes → cria índice → EXPLAIN depois. **Resultado honesto:** no DuckDB (motor
colunar/vetorizado) o plano NÃO muda e não há ganho — o índice B-tree/ART não é acionado
em varreduras analíticas. Índice ajuda em bancos orientados a linha (Postgres/SQLite).
A técnica que de fato acelerou aqui foi o particionamento (A.7).

In [8]:
q = '''SELECT d.mes, SUM(f.qtd_transactions) tx FROM fato_atividade f
       JOIN dim_data d ON f.sk_data=d.sk_data WHERE d.mes=7 GROUP BY d.mes'''
print("=== EXPLAIN ANTES do índice ==="); print(con.execute("EXPLAIN "+q).fetchall()[0][1][:500])
con.execute("CREATE INDEX idx_fato_data ON fato_atividade(sk_data)")
con.execute("CREATE INDEX idx_fato_prod ON fato_atividade(sk_produto)")
print("\n=== EXPLAIN DEPOIS do índice ==="); print(con.execute("EXPLAIN "+q).fetchall()[0][1][:500])
print("\nConclusão: plano idêntico — índice não acelera varredura analítica no DuckDB.")
logar("A.6 - Índices")

=== EXPLAIN ANTES do índice ===
┌───────────────────────────┐
│         PROJECTION        │
│    ────────────────────   │
│__internal_decompress_integ│
│     ral_integer(#0, 7)    │
│             #1            │
│                           │
│          ~6 rows          │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│   PERFECT_HASH_GROUP_BY   │
│    ────────────────────   │
│         Groups: #0        │
│    Aggregates: sum(#1)    │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         PROJECTION

=== EXPLAIN DEPOIS do índice ===
┌───────────────────────────┐
│         PROJECTION        │
│    ────────────────────   │
│__internal_decompress_integ│
│     ral_integer(#0, 7)    │
│             #1            │
│                           │
│          ~6 rows          │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│   PERFECT_HASH_GROUP_BY   │
│    ────────────────────   │
│         Groups: #0        │
│    Aggregates: sum(#1)    │
└────────────

## A.7 — TÉCNICA 3: Particionamento (OBRIGATÓRIA)
Exporta a fato em Parquet particionado por **ano/mês** (o dataset é todo de 2015, então
particionar só por ano daria 1 partição; por mês há partition pruning real).
**Entrega:** mostra a estrutura de pastas e compara ler 1 mês vs ler tudo.

In [9]:
shutil.rmtree('dw_export', ignore_errors=True); os.makedirs('dw_export', exist_ok=True)
con.execute('''COPY (SELECT f.*, d.ano, d.mes FROM fato_atividade f
                JOIN dim_data d ON f.sk_data=d.sk_data)
               TO 'dw_export' (FORMAT PARQUET, PARTITION_BY (ano,mes), OVERWRITE_OR_IGNORE)''')
parts=[os.path.relpath(os.path.join(r,f),'dw_export') for r,_,fs in os.walk('dw_export') for f in fs if f.endswith('.parquet')]
print("Estrutura de partições:"); [print("  ",p) for p in sorted(parts)]
n_tudo=con.execute("SELECT COUNT(*) FROM read_parquet('dw_export/**/*.parquet',hive_partitioning=true)").fetchone()[0]
n_mes=con.execute("SELECT COUNT(*) FROM read_parquet('dw_export/ano=2015/mes=7/*.parquet')").fetchone()[0]
print(f"\nLer TODAS as partições: {n_tudo:,} linhas")
print(f"Ler só mês=7 (pruning):  {n_mes:,} linhas  <- lê muito menos dado")
logar("A.7 - Particionamento")

Estrutura de partições:
   ano=2015\mes=5\data_0.parquet
   ano=2015\mes=6\data_0.parquet
   ano=2015\mes=7\data_0.parquet
   ano=2015\mes=8\data_0.parquet
   ano=2015\mes=9\data_0.parquet

Ler TODAS as partições: 2,271,479 linhas
Ler só mês=7 (pruning):  578,665 linhas  <- lê muito menos dado


## A.8 — Analytics no DW (5 perguntas)
Cada consulta usa JOIN com ≥1 dimensão. A Pergunta 2 usa a dimensão SCD Tipo 2
(histórico vs atual). A Pergunta 4 usa Window Function.

In [10]:
print("P1 — Eventos (views) por mês:")
display(con.execute('''SELECT d.ano,d.mes,SUM(f.qtd_views) AS total_views FROM fato_atividade f
   JOIN dim_data d ON f.sk_data=d.sk_data GROUP BY d.ano,d.mes ORDER BY d.ano,d.mes''').df())

print("P2 — SCD Tipo 2: produtos com categoria histórica vs atual:")
display(con.execute('''SELECT id_natural,
     MAX(CASE WHEN flag_atual=0 THEN categoria END) categoria_antiga,
     MAX(CASE WHEN flag_atual=1 THEN categoria END) categoria_atual
   FROM dim_produto GROUP BY id_natural HAVING COUNT(*)>1 LIMIT 10''').df())

print("P3 — Transações por categoria (top 10):")
display(con.execute('''SELECT c.categoria, SUM(f.qtd_transactions) tx FROM fato_atividade f
   JOIN dim_categoria c ON f.sk_categoria=c.sk_categoria GROUP BY c.categoria
   ORDER BY tx DESC LIMIT 10''').df())

print("P4 — Window Function: ranking de meses por views:")
display(con.execute('''SELECT d.mes, SUM(f.qtd_views) AS total_views,
     RANK() OVER (ORDER BY SUM(f.qtd_views) DESC) ranking
   FROM fato_atividade f JOIN dim_data d ON f.sk_data=d.sk_data
   GROUP BY d.mes ORDER BY ranking''').df())

print("P5 — Taxa de conversão (transactions / views) por mês:")
display(con.execute('''SELECT d.mes, SUM(f.qtd_views) AS total_views, SUM(f.qtd_transactions) tx,
     ROUND(SUM(f.qtd_transactions)*100.0/NULLIF(SUM(f.qtd_views),0),3) taxa_pct
   FROM fato_atividade f JOIN dim_data d ON f.sk_data=d.sk_data GROUP BY d.mes ORDER BY d.mes''').df())
logar("A.8 - Analytics")

P1 — Eventos (views) por mês:


,ano,mes,total_views
0,2015,5,571672.0
1,2015,6,590255.0
2,2015,7,674820.0
3,2015,8,533905.0
4,2015,9,293660.0


P2 — SCD Tipo 2: produtos com categoria histórica vs atual:


,id_natural,categoria_antiga,categoria_atual
0,112854,5,7
1,186360,11,13
2,307030,31,33
3,250880,31,33
4,442346,47,49
5,196303,4,6
6,337767,18,20
7,195315,16,18
8,38175,26,28
9,345529,30,32


P3 — Transações por categoria (top 10):


,categoria,tx
0,31,553.0
1,35,548.0
2,37,540.0
3,18,515.0
4,4,504.0
5,34,504.0
6,44,501.0
7,29,501.0
8,39,499.0
9,45,499.0


P4 — Window Function: ranking de meses por views:


,mes,total_views,ranking
0,7,674820.0,1
1,6,590255.0,2
2,5,571672.0,3
3,8,533905.0,4
4,9,293660.0,5


P5 — Taxa de conversão (transactions / views) por mês:


,mes,total_views,tx,taxa_pct
0,5,571672.0,4611.0,0.807
1,6,590255.0,5043.0,0.854
2,7,674820.0,5802.0,0.860
3,8,533905.0,4632.0,0.868
4,9,293660.0,2369.0,0.807


# TRILHA B — DATA LAKE (ELT / Medallion)
## B.1 Bronze · B.2 Silver

In [11]:
# Bronze: bruto + metadados
l1.to_csv('datalake/bronze/lote1.csv', index=False); l2.to_csv('datalake/bronze/lote2.csv', index=False)
# Silver: limpa, deduplica, colunas calculadas
silver=pd.concat([l1,l2[l1.columns]],ignore_index=True).drop_duplicates()
silver=silver.dropna(subset=['timestamp','itemid','event'])
silver['ano']=silver['timestamp'].dt.year; silver['mes']=silver['timestamp'].dt.month
silver['dia']=silver['timestamp'].dt.day
silver.to_csv('datalake/silver/eventos_silver.csv', index=False)
print(f"Bronze: {len(l1)+len(l2):,}  ->  Silver (dedup): {len(silver):,}  (removidos {len(l1)+len(l2)-len(silver):,})")
logar("B.1/B.2 - Bronze/Silver", silver=len(silver))

Bronze: 2,756,101  ->  Silver (dedup): 2,755,641  (removidos 460)


## B.3 — TÉCNICA 4: Particionamento físico Hive-style (OBRIGATÓRIA)
Silver salva em `silver/particionado/ano=2015/mes=N/`. Partition pruning: filtrar 1 mês
lê só a pasta daquele mês.

In [12]:
base='datalake/silver/particionado'; shutil.rmtree(base,ignore_errors=True)
for (a,m),g in silver.groupby(['ano','mes']):
    os.makedirs(f'{base}/ano={a}/mes={m}',exist_ok=True)
    g.to_parquet(f'{base}/ano={a}/mes={m}/data.parquet', index=False)
import glob
print("Partições:"); [print("  ",p) for p in sorted(glob.glob(f'{base}/ano=*/mes=*'))]
tudo=duckdb.sql(f"SELECT COUNT(*) FROM read_parquet('{base}/**/*.parquet',hive_partitioning=true)").fetchone()[0]
um=duckdb.sql(f"SELECT COUNT(*) FROM read_parquet('{base}/ano=2015/mes=7/*.parquet')").fetchone()[0]
print(f"\nSem partição (lê tudo): {tudo:,} linhas | Com pruning (mês=7): {um:,} linhas")
logar("B.3 - Particionamento físico")

Partições:
   datalake/silver/particionado\ano=2015\mes=5
   datalake/silver/particionado\ano=2015\mes=6
   datalake/silver/particionado\ano=2015\mes=7
   datalake/silver/particionado\ano=2015\mes=8
   datalake/silver/particionado\ano=2015\mes=9

Sem partição (lê tudo): 2,755,641 linhas | Com pruning (mês=7): 697,849 linhas


## B.4 — TÉCNICA 5: Time Travel com Delta Lake (OBRIGATÓRIA)
Lote 1 → versão 0 (overwrite). Lote 2 → versão 1 (append). Consultamos `dt.history()`
e lemos a versão antiga.
**Para que serve no mundo real:** auditoria e rollback — voltar a uma versão anterior
se uma carga vier corrompida, sem perder o passado.

In [15]:
import tempfile
# OBS: o Delta Lake faz cada commit gravando um arquivo temporario e renomeando
# de forma atomica. O Google Drive ("G:\\Meu Drive") nao suporta esse rename e
# quebra com OSError 1 ("Funcao incorreta"). Por isso a tabela Delta vai para um
# disco local de verdade (pasta temp do sistema). O conteudo e o mesmo.
DELTA_PATH = os.path.join(tempfile.gettempdir(), 'dw_projeto_delta', 'eventos')
shutil.rmtree(DELTA_PATH, ignore_errors=True)
shutil.rmtree('delta/eventos', ignore_errors=True)

l1d=l1.copy(); l1d['timestamp']=l1d['timestamp'].astype(str)
l2d=l2.copy(); l2d['timestamp']=l2d['timestamp'].astype(str)
write_deltalake(DELTA_PATH, l1d, mode='overwrite')                          # v0
write_deltalake(DELTA_PATH, l2d, mode='append', schema_mode='merge')        # v1
dt=DeltaTable(DELTA_PATH)
print(f"Delta em: {DELTA_PATH}")
print(f"Versão 0: {len(DeltaTable(DELTA_PATH,version=0).to_pandas()):,} linhas")
print(f"Versão 1 (atual): {len(dt.to_pandas()):,} linhas")
print("\nhistory():")
for h in dt.history(): print("   versão",h.get('version'),"| operação",h.get('operation'))
logar("B.4 - Time Travel")

Delta em: C:\Users\aquil\AppData\Local\Temp\dw_projeto_delta\eventos
Versão 0: 2,480,490 linhas
Versão 1 (atual): 2,756,101 linhas

history():
   versão 1 | operação WRITE
   versão 0 | operação WRITE


## B.5 — TÉCNICA 6: Schema Evolution (OBRIGATÓRIA)
A coluna `canal` só existe no Lote 2. Com `schema_mode='merge'` o Delta aceita a evolução
sem quebrar. Comparamos o schema da v0 com o da v1.

In [16]:
cols_v0=[f.name for f in DeltaTable(DELTA_PATH,version=0).schema().fields]
cols_v1=[f.name for f in DeltaTable(DELTA_PATH).schema().fields]
print("Schema v0:", cols_v0)
print("Schema v1:", cols_v1)
print("Coluna nova que passou a existir sem quebrar:", set(cols_v1)-set(cols_v0))
logar("B.5 - Schema Evolution")

Schema v0: ['timestamp', 'visitorid', 'event', 'itemid', 'transactionid', '_ingested_at', '_source_file', '_batch_id']
Schema v1: ['timestamp', 'visitorid', 'event', 'itemid', 'transactionid', '_ingested_at', '_source_file', '_batch_id', 'canal']
Coluna nova que passou a existir sem quebrar: {'canal'}


## B.6 Gold · B.7 Analytics na Gold + validação cruzada

In [17]:
g_mensal=silver.groupby(['ano','mes']).size().reset_index(name='qtd_eventos')
g_evento=silver.groupby('event').size().reset_index(name='qtd_eventos')
os.makedirs('datalake/gold',exist_ok=True)
g_mensal.to_csv('datalake/gold/gold_mensal.csv',index=False)
g_evento.to_csv('datalake/gold/gold_evento.csv',index=False)
display(g_evento)

# Validação cruzada HONESTA
total_dw=con.execute("SELECT SUM(qtd_views+qtd_addtocart+qtd_transactions) FROM fato_atividade").fetchone()[0]
total_gold=int(g_evento['qtd_eventos'].sum())
print(f"\nDW (fato):  {int(total_dw):,}")
print(f"Gold:        {total_gold:,}")
print(f"Diferença:   {abs(int(total_dw)-total_gold):,}")
print("Causa: a Silver/Gold removem duplicatas e nulos; o fato do DW parte do bruto.")
print("=> A diferença é esperada e explicável, não um erro.")
logar("B.6/B.7 - Gold")

,event,qtd_eventos
0,addtocart,68966
1,transaction,22457
2,view,2664218



DW (fato):  2,756,101
Gold:        2,755,641
Diferença:   460
Causa: a Silver/Gold removem duplicatas e nulos; o fato do DW parte do bruto.
=> A diferença é esperada e explicável, não um erro.


# Monitoramento do pipeline

In [18]:
with open('logs/pipeline_log.json','w') as f: json.dump(log_pipeline,f,indent=2,default=str)
print(f"Total de etapas: {len(log_pipeline['etapas'])} | todas OK")
print("Log salvo em logs/pipeline_log.json")
con.close()

Total de etapas: 14 | todas OK
Log salvo em logs/pipeline_log.json


# Conclusão — ETL vs ELT
**a) Mais trabalhosa:** a Trilha A (ETL). Exige modelar o Star Schema, SKs, SCD Tipo 2 e
integridade antes de carregar. O ELT joga o bruto na Bronze e transforma depois.

**b) DW e Gold batem?** Quase: DW parte do bruto, Gold deduplica — diferença explicável
(limpeza), não erro.

**c) Dados novos amanhã:** ELT reexecuta mais fácil — só anexar na Bronze.

**d) SCD Tipo 2 ajudou mais no DW**, onde o histórico fica estruturado e consultável.

**e) Time Travel vs SCD Tipo 2:** SCD Tipo 2 para histórico de um atributo de dimensão
consultável por período; Time Travel para auditoria/rollback da tabela inteira.

**Recomendação:** arquitetura híbrida — Data Lake (Bronze/Silver/Gold + Delta com time
travel) para ingestão flexível e auditoria; DW em Star Schema com SCD Tipo 2 e
particionamento para análise estruturada e rápida.